<a href="https://colab.research.google.com/github/karquiza/bikeshare-toronto-demand/blob/main/notebooks/M4_Report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MAI 103 Group 11 Assignment 2 - Milestone 4: Baseline Modeling, Evaluation & Data Generation Strategy

## Bike Share Station Demand for Growth Planning

### Members
- TODO: Add members here when ready to submit

## Part 1: Experimental Design & Data Preparation

### 1.1 Modeling Objective

**Problem Formulation:** Can we predict the hourly departures and arrivals at bike stations and classify whether the demand for a station is Low, Medium, or High, so that Bike Share Toronto can plan where to add new docks or stations?

**Target Variables:**

1. **Regression:** Departures per station per hour, Arrivals per station per hour during the peak months (May to October).
- For each month, a station is considered as open/operating if there is at least 1 trip starting at that station.
- For open/operating stations, hours with zero trips (ex. typical of night time hours) are included in the model.
- In Milestone 3, we set the following flags in the cleaned data to mark valid or invalid trips:
    - `flag_short`: Trip duration is under 60 seconds
    - `flag_long`: Trip duration is over 6 hours
    - `flag_no_end`: The trip has no `End_Station_Id`
    - `is_valid`: True when none of the flags is set, False if at least one is set
- (a) Departures - Valid trips (rows) have no `flag_short` or `flag_long` set. A missing end station doesn't matter.
- (b) Arrivals - Valid trips (rows) have none of the flags (`flag_short`, `flag_long`, `flag_no_end`) set.
    - Note: We said in Milestone 3 that we would no longer include Arrivals as a target variable due to the missing `End_Station_Id`s. However, after discussion, we have decided that the small percentage of missing `End_Station_Id`s should not affect the model too much, and this target variable is relevant in determining how busy a station is.

2. **Classification:** Predict the demand of stations during May-October 2025 as Low, Medium, or High, based on the training data from the same months in 2023 and 2024.
- One row is one station in one month. A station-month exists if the station had at least 1 valid departure in that month.
- Demand rate for a station is calculated by the average valid departures per operating day for this version.
    - An operating day is all the days in an operating month (if there is at least 1 valid departure in that month, all days in that month will be considered operating days). This is the same rule as we used for the Regression task, and we keep it here for consistency.
    - Days with no trips in an operating month will have a zero demand.
    - Valid departures use the same `is_valid` flag as in Milestone 3. Arrivals are not included in this rate.
    - Note: There are changes from Milestone 3, namely changing the definition to use operating days instead of the active days, because using active days only hides low-demand stations that may have few active days.
- The thresholds for the classification are the following:
    - **TODO: Add code to recompute target variables here. Low=lowest 1/3, Medium=middle 1/3, High=highest 1/3 cutoffs based on the training data only.**
- Stations that first appear in 2025 have no 2023 to 2024 rate and are not in the classification data. They are used only in the business check that compares 2024 High stations with the new 2025 stations.
    

**Features:**

Each ML task has a different set of features.

1. **Regression**
- One row per station per hour
- (a) Features from date/timestamp:

     - `hour_of_day` - categorical values: hr_0-hr_23, rush hour will have different behaviour
     - `day_of_week` - categorical values: Monday(0)-Sunday(6), riding patterns are different on weekdays and weekends
     - `weekend_or_holiday` - flag (1 if weekend or holiday, 0 otherwise)
     - `year` - numerical values: 2023, 2024, 2025
     - `month` - categorical values: May-Oct

- (b) Features from weather (TBD)

    - `prec_mm_hr` - numerical value, precipitation/rain in that hour
    - `mean_temp_c_hr` - numerical value, temperature in that hour
    - `prec_mm_prev_hr` - numerical value, precipitation/rain in the previous hour (since people check weather before deciding to ride)

- (c) Features from station:

    - `station_id` - categorical
    - `new_station` - flag (1 if first month that the station is seen, 0 otherwise). We are doing regression on May to October months only, but we have data from January 2023. We will check for the first occurrence of a station since the very beginning of our data (January 2023) in order to set this flag. (Note that this information is in Milestone 3, since we cleaned the data using the entire dataset)


2. **Classification**

- One row per station per month (May to October)
- (a) Features from month's activity:
     - `monthly_volume` - numerical value, valid departures from the station in that month.
      - `was_flagged_new` - flag (1 if that month is the first month the station appears in the data, 0 otherwise). `is_new_start_station` flag from Milestone 3 on monthly level.

- (b) Features from station history (Milestone 3 station months)
    - `first_year` - numerical value, the year the station first appears in the data.
    - `first_month` - numerical value, the month the station first appears in the data.

### 1.2 Evaluation Criteria aligned with the Problem

Our training-validation-test split has been changed since Milestone 3:
- Training: May-October 2023 + May 2024
- Validation: June-October 2024
- Test: May-October 2025

We will score each model using the Validation set to help choose the best settings (hyperparameter tuning) and models. Once a model and/or its hyperparameter settings have been chosen, we will rerun the training on the Training+Validation set and evaluate this final model on the Test set once at the end.

1. **Regression** (departures per hour, arrivals per hour)
- Performance metrics: MAE, RMSE for departures and arrivals separately, since they are separate models.
- Baseline to compare against: average departures/arrivals for the same station, hour of day, and day of week in the training months.
- Success criteria: MAE on Test set should be lower than the baseline for both departures and arrivals.

2. **Classification**
- Performance metrics: TODO based on the Classification notebooks

3. **Business check**
- Compare the stations classified as High in 2024 with the stations that first appear in 2025, to see whether Bike Share Toronto's expansion followed the high-demand stations. This is to check whether our project's outputs would be useful for growth planning, according to the problem formulation we were trying to solve.

### 1.3 Preprocessing

This section builds the feature tables for the regression and classification models:
1. **Station-hour table** - For regression tasks (departures and arrivals): 1 row per station per hour. Features are departures, arrivals, time/calendar-related, weather, and new-station flag to mark if a station is new for that month.
2. **Station-month table** -- For classification task: 1 row per station per month. Features include **TODO: add based on classification notebook**

#### 1.3.1 Trip data

1. **Limit to peak season only (May to October months)** - Bike share demand is seasonal as we observed in Milestone 3 exploration. Since our business problem is to use our model to help determine areas of growth where new docks or bikes may be placed, we will limit our trip data to the most active summer months and include the months where activity begins to increase more each year as well we the months after summer when bike ridership slows down.

2. **Trip Validity** - Use the flags defined in Milestone 3: `flag_short`, `flag_long`, `flag_no_end`, and `is_valid`. (Refer to 1.1 Target Variables for the same definitions in this notebook) Since we are predicting departures and arrivals separately, counted trips for each regression task use the flags differently:
- **Departures** Using the start station: a trip counts as valid if it has neither `flag_short` nor `flag_long`. Missing end station doesn't matter for departures.
- **Arrivals** Using the End station: a trip counts as valid only if `is_valid` is true (this flag considers the `flag_no_end` as well as the other 2 flags and is true only if none of them are set).

In [4]:
# Read file and set up dataframe

import os, glob, re, time
import numpy as np
import pandas as pd

# 1. Setup and paths (same pattern as the other notebooks)
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = "/content/drive/MyDrive/DataSets"
    REFERENCE = f"{DATA}/reference"
    WEATHER_XLSX = f"{DATA}/bike_stations_hourly_weather_analyzed.xlsx"  # output of 03_testing_code.ipynb
else:
    DATA = "../data"
    REFERENCE = "../reference"
    WEATHER_XLSX = f"{DATA}/external/bike_stations_hourly_weather_analyzed.xlsx"

CLEAN = f"{DATA}/clean"
AGG = f"{DATA}/aggregate"
os.makedirs(AGG, exist_ok=True)

YEARS = [2023, 2024, 2025]
MONTHS = list(range(5, 11)) # May to October

def clean_file(year, month):
    return f"{CLEAN}/clean_ridership_{year}_{month:02d}.csv"

print("Clean files found:", sum(os.path.exists(clean_file(y, m)) for y in YEARS for m in MONTHS), "of", len(YEARS) * len(MONTHS))


USECOLS = ["Trip_Id", "Start_Time", "End_Time", "Start_Station_Id", "End_Station_Id",
           "flag_short", "flag_long", "flag_no_end", "is_valid"]

def read_clean(path):
    df = pd.read_csv(path, usecols=USECOLS, parse_dates=["Start_Time", "End_Time"],
                     dtype={"Start_Station_Id": "Int64", "End_Station_Id": "Int64"})
    df["start_hour"] = df["Start_Time"].dt.floor("h")
    df["end_hour"] = df["End_Time"].dt.floor("h")
    return df

## Aggregate departures and arrivals per station per hour in one month
def aggregate_month(df):
    dep_rows = df[~(df["flag_short"] | df["flag_long"])]
    dep = (dep_rows.groupby(["Start_Station_Id", "start_hour"]).size().rename("departures")
           .reset_index().rename(columns={"Start_Station_Id": "station_id", "start_hour": "hour_ts"}))
    arr_rows = df[df["is_valid"]]
    arr = (arr_rows.groupby(["End_Station_Id", "end_hour"]).size().rename("arrivals")
           .reset_index().rename(columns={"End_Station_Id": "station_id", "end_hour": "hour_ts"}))
    out = dep.merge(arr, on=["station_id", "hour_ts"], how="outer")
    out["departures"] = out["departures"].fillna(0).astype("int64")
    out["arrivals"] = out["arrivals"].fillna(0).astype("int64")
    return out

# Check on one month: the counts must add up to the trips we meant to count
jul23 = read_clean(clean_file(2023, 7))
sh = aggregate_month(jul23)
print(f"JULY 2023 Check - Trips in: {len(jul23):,}   station-hours out: {len(sh):,}")
print("Departures counted:", f"{sh['departures'].sum():,}", " trips not short/long:", f"{(~(jul23['flag_short'] | jul23['flag_long'])).sum():,}")
print("Arrivals counted:  ", f"{sh['arrivals'].sum():,}", " trips valid:         ", f"{jul23['is_valid'].sum():,}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Clean files found: 18 of 18
JULY 2023 Check - Trips in: 735,924   station-hours out: 288,233
Departures counted: 733,606  trips not short/long: 733,606
Arrivals counted:   733,428  trips valid:          733,428


#### 1.3.2 Station-hour table

1. **Include Zero-trip hours** - The group agreed to include hours with zero trips in operating months (ex. nighttime hours) because excluding them may give the model incorrect information about the true demand of a station. We build the the station-hour table for operating stations for every hour of the month. If there are missing hours, we assume that they are missing because there are no trips recorded in the dataset and the counts will be set to 0. (Same approach as Milestone 3. For this Milestone, we apply the same approach to arrivals.)

2. **Add Time-related columns from timestamps** - These are the following features needed for the regression tasks: `hour_of_day`, `day_of_week`, `year`, `month`. The `weekend_or_holiday` flag and other features not created directly from the timestamps will be added later.

In [6]:
# Zero-trip hours
def add_zerotrip_hours(sh, year, month):
    hours = pd.date_range(f"{year}-{month:02d}-01", periods=pd.Period(f"{year}-{month:02d}").days_in_month * 24, freq="h")
    # trips can end in the next month; keep only hours inside this month so months do not overlap
    sh = sh[(sh["hour_ts"] >= hours[0]) & (sh["hour_ts"] <= hours[-1])]
    active = sh["station_id"].dropna().unique()
    grid = pd.MultiIndex.from_product([active, hours], names=["station_id", "hour_ts"]).to_frame(index=False)
    out = grid.merge(sh, on=["station_id", "hour_ts"], how="left")
    out["departures"] = out["departures"].fillna(0).astype("int64")
    out["arrivals"] = out["arrivals"].fillna(0).astype("int64")
    return out

full = add_zerotrip_hours(sh, 2023, 7)
print(f"Operating stations: {full['station_id'].nunique():,}  x  hours: {full['hour_ts'].nunique()}  =  rows: {len(full):,}")
print(f"Percentage of station-hours with zero departures: {(full['departures'] == 0).mean()*100:.1f}%")

# Time-related columns from timestamps
def add_calendar(df):
    ts = df["hour_ts"]
    df["hour_of_day"] = ts.dt.hour
    df["day_of_week"] = ts.dt.dayofweek
    df["year"] = ts.dt.year
    df["month"] = ts.dt.month
    return df

# Loop over May to October 2023 to 2025, check the counts, keep the tables for merging with weather data
# Note: The 2025 months (test set) go through the same preprocessing code but are not shown shown
year_parts, check_rows = {}, []
for y in YEARS:
    for m in MONTHS:
        trips = read_clean(clean_file(y, m))
        sh = add_calendar(add_zerotrip_hours(aggregate_month(trips), y, m))
        year_parts.setdefault(y, []).append(sh)
        check_rows.append({"year": y, "month": m,
                           "trips_for_departures": int((~(trips["flag_short"] | trips["flag_long"])).sum()),
                           "departures_in_table": int(sh["departures"].sum()),
                           "trips_valid": int(trips["is_valid"].sum()),
                           "arrivals_in_table": int(sh["arrivals"].sum()),
                           "stations": sh["station_id"].nunique(), "rows": len(sh)})
        if y < 2025:
            print(f"{y}-{m:02d}: {len(sh):>9,} station-hours, {sh['station_id'].nunique()} stations")

checks = pd.DataFrame(check_rows)
checks["departures_match"] = checks["trips_for_departures"] == checks["departures_in_table"]
checks["arrivals_in_next_month"] = checks["trips_valid"] - checks["arrivals_in_table"]
checks.to_csv(f"{AGG}/station_hour_checks.csv", index=False)

shown = checks[checks["year"] < 2025]
print("\nDepartures match the clean trip counts in every training and validation month:", shown["departures_match"].all())
print("Arrivals that fell into the next month:", f"{shown['arrivals_in_next_month'].sum():,}",
      f"of {shown['trips_valid'].sum():,} valid trips ({shown['arrivals_in_next_month'].sum()/shown['trips_valid'].sum()*100:.4f}%)")
shown

Operating stations: 681  x  hours: 744  =  rows: 506,664
Percentage of station-hours with zero departures: 54.2%
2023-05:   499,224 station-hours, 671 stations
2023-06:   483,840 station-hours, 672 stations
2023-07:   506,664 station-hours, 681 stations
2023-08:   518,568 station-hours, 697 stations
2023-09:   509,760 station-hours, 708 stations
2023-10:   546,096 station-hours, 734 stations
2024-05:   595,944 station-hours, 801 stations
2024-06:   586,080 station-hours, 814 stations
2024-07:   614,544 station-hours, 826 stations
2024-08:   627,192 station-hours, 843 stations
2024-09:   615,600 station-hours, 855 stations
2024-10:   634,632 station-hours, 853 stations

Departures match the clean trip counts in every training and validation month: True
Arrivals that fell into the next month: 1,830 of 9,033,231 valid trips (0.0203%)


,year,month,trips_for_departures,departures_in_table,trips_valid,arrivals_in_table,stations,rows,departures_match,arrivals_in_next_month
0,2023,5,587581,587581,587456,587330,671,499224,True,126
1,2023,6,661666,661666,661554,661339,672,483840,True,215
2,2023,7,733606,733606,733428,733298,681,506664,True,130
3,2023,8,757479,757479,757178,757016,697,518568,True,162
4,2023,9,751556,751556,751190,751024,708,509760,True,166
5,2023,10,598812,598812,598533,598479,734,546096,True,54
6,2024,5,684185,684185,683735,683527,801,595944,True,208
7,2024,6,764575,764575,763955,763802,814,586080,True,153
8,2024,7,902567,902567,902081,901893,826,614544,True,188
9,2024,8,904255,904255,903718,903497,843,627192,True,221


#### 1.3.3 Weather Data

**TODO: Writeup and code about the weather data**: Data source and weather stations, why Toronto City only, hourly download for May-October 2023-2025, and output file for 1.3.4.

1. **Source:** Environment and Climate Change Canada hourly readings, one row per hour (note that the time is in Local Standard Time and not Toronto time). Fields we kept: temperature, precipitation, etc. (please check this)
2. **Weather Stations used:** Considered Toronto City (31688), Toronto City Centre (48549), Toronto Island, Pearson. Which one we chose and why.

In [ ]:
#**3. TODO: Add the download code here, but add a check if the output file already exists or not:

if os.path.exists(WEATHER_XLSX):
    print("Weather file exists, download skipped: " , WEATHER_XLSX)
else:
    # Code to download goes here

4. Add short description of what the weather file contains.

#### 1.3.4 Weather join

## Part 2: Data Generation / Augmentation (If Applicable)

Our project does not need data generation or augmentation.
1. For Regression tasks - Even though there are many zero-trip hours in the feature set, that is an actual observation for those hours for nighttime or for very low-demand stations. The zeros are real information about those hours and stations and it is useful for the business problem (planning based on how much demand is at stations). The prediction should show zero as well for those hours.
2. For Classification task - We defined the Low, Medium, and High cutoffs as the lower, middle, and upper thirds of the training data, so classes are closely equal in size (no class imbalance).
3. The dataset is already several million rows, so there is more than enough data and we don't need to generate more synthetic data for training or test.
4. Stations added in 2025 may cause problems for our models since they would not exist at all in the training or validation data. For the classification task, the group decided to exclude these stations and use them in the business check instead (see Evaluation Criteria).

## Part 3: Final Model Development & Optimization

## Part 4: Model Evaluation & Analytical Depth

## Part 5: Analytical Reflection (1 Page Required-Very important)